# Train the card detector

Choose axis boxes, oriented boxes, or four corners with a shared frozen DINOv3 ConvNeXt Tiny backbone. Set `fast_dev_run = False` for full training.


In [ ]:
import json
from pathlib import Path

import cv2
import lightning.pytorch as pl
import torch
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from set_master.training.data import CardDataModule
from set_master.training.model import DEFAULT_REPO, DEFAULT_WEIGHTS
from set_master.training.config import DetectorConfig, DetectorVariant
from set_master.training.models.factory import create_detector, load_detector
from set_master.training.evaluation import prediction_overlay
from set_master.paths import PROJECT_ROOT, RUNS_DIR, SYNTHETIC_DATASETS_DIR
from set_master.visualization import display_image

print(f'Project: {PROJECT_ROOT}')
print(f'CUDA available: {torch.cuda.is_available()}')


## Settings

Set `dataset_run` to the run you want to train on. With `val_data = None`, 20% of images are reserved for validation.

Choose `variant` and an `experiment_name`. Geometry experiments require corner and oriented-box annotations.


In [ ]:
variant = DetectorVariant.BOX  # BOX, ORIENTED, or CORNERS
experiment_name = 'YOUR_EXPERIMENT'
dataset_run = 'run_20261010_125004_004209'
train_data = SYNTHETIC_DATASETS_DIR / dataset_run
val_data = None
val_fraction = 0.2

weights_path = DEFAULT_WEIGHTS
repo_path = DEFAULT_REPO
output_dir = RUNS_DIR / 'card_detector' / variant.value / experiment_name
resume_checkpoint = None

batch_size = 2
num_workers = 0
epochs = 20
learning_rate = 1e-3
weight_decay = 1e-4
min_size = 640
max_size = 1280
accelerator = 'gpu'
precision = '32-true'
seed = 42
fast_dev_run = False


## Data preview

Ground-truth boxes are green.


In [ ]:
if not (train_data / 'annotations.json').is_file():
    raise FileNotFoundError('Choose a generated dataset in train_data, or create one with generate_synthetic_dataset.ipynb.')
if batch_size < 1 or num_workers < 0 or not 0 < min_size <= max_size:
    raise ValueError('Require batch_size > 0, num_workers >= 0 and 0 < min_size <= max_size.')

pl.seed_everything(seed, workers=True)
data = CardDataModule(
    train_root=train_data, val_root=val_data, batch_size=batch_size,
    num_workers=num_workers, val_fraction=val_fraction, seed=seed, variant=variant,
)
data.setup('fit')
print(f'Training images: {len(data.train_dataset)}')
print(f'Validation images: {len(data.val_dataset)}')

sample_image, sample_target = data.train_dataset[0]
preview = cv2.cvtColor(sample_image.mul(255).byte().permute(1, 2, 0).numpy(), cv2.COLOR_RGB2BGR)
for x1, y1, x2, y2 in sample_target['boxes'].int().tolist():
    cv2.rectangle(preview, (x1, y1), (x2 - 1, y2 - 1), (0, 180, 0), 2)
display_image(preview)


## Model


In [ ]:
model = create_detector(
    DetectorConfig(variant, learning_rate, weight_decay, min_size, max_size),
    weights_path=weights_path, repo_path=repo_path,
    initialize_pretrained=resume_checkpoint is None,
)
trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
frozen = sum(parameter.numel() for parameter in model.parameters() if not parameter.requires_grad)
print(f'Trainable parameters: {trainable:,}')
print(f'Frozen parameters: {frozen:,}')


## Logging and checkpoints


In [ ]:
logger = CSVLogger(str(output_dir), name='')
logger.log_hyperparams(model.hparams)
run_dir = Path(logger.log_dir)
run_dir.mkdir(parents=True, exist_ok=True)
config = {
    'variant': variant.value, 'experiment_name': experiment_name,
    'train_data': str(train_data), 'val_data': str(val_data) if val_data else None,
    'val_fraction': val_fraction, 'weights_path': str(weights_path), 'repo_path': str(repo_path),
    'batch_size': batch_size, 'num_workers': num_workers, 'epochs': epochs,
    'learning_rate': learning_rate, 'weight_decay': weight_decay,
    'min_size': min_size, 'max_size': max_size, 'accelerator': accelerator,
    'precision': precision, 'seed': seed, 'fast_dev_run': fast_dev_run,
    'resume_checkpoint': str(resume_checkpoint) if resume_checkpoint else None,
}
(run_dir / 'config.json').write_text(json.dumps(config, indent=2) + '\n')
(run_dir / 'split.json').write_text(json.dumps(data.split_manifest, indent=2) + '\n')

checkpoint = ModelCheckpoint(
    dirpath=run_dir / 'checkpoints', monitor='val/map' if variant == DetectorVariant.BOX else 'val/geometry_map', mode='max',
    save_top_k=1, save_last=True, filename='epoch-{epoch:02d}', auto_insert_metric_name=False,
)
trainer = pl.Trainer(
    max_epochs=epochs, accelerator=accelerator, devices=1, precision=precision,
    logger=logger, callbacks=[checkpoint], gradient_clip_val=1.0,
    log_every_n_steps=1, fast_dev_run=fast_dev_run, default_root_dir=str(output_dir),
)
print(f'Run directory: {run_dir}')
print('Mode:', 'one-batch smoke check' if fast_dev_run else 'full training')


## Train


In [13]:
trainer.fit(model, datamodule=data, ckpt_path=resume_checkpoint)

print('Final metrics:')
for name, value in trainer.callback_metrics.items():
    print(f'  {name}: {value.item():.4f}')
if checkpoint.best_model_path:
    print(f'Best checkpoint: {checkpoint.best_model_path}')
    print(f'Last checkpoint: {checkpoint.last_model_path}')


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type                 | Params | Mode  | FLOPs
------------------------------------------------------------------
0 | detector | FasterRCNN           | 45.0 M | train | 0    
1 | box_map  | MeanAveragePrecision | 0      | train | 0    
------------------------------------------------------------------
17.2 M    Trainable params
27.8 M    Non-trainable params
45.0 M    Total params
180.200   Total estimated model params size (MB)
40        Modules in train mode
151       Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/florian/src/python/set-master/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/florian/src/python/set-master/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/florian/src/python/set-master/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/florian/src/python/set-master/.venv/lib/python3.13/site-packages/lightning/pytorch/loops/fit_loop.p

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.


Final metrics:
  train/loss: 0.0670
  train/loss_step: 0.0700
  val/map: 0.8935
  val/map_50: 1.0000
  val/map_75: 1.0000
  val/mar_100: 0.9252
  train/loss_epoch: 0.0670
  train/loss_classifier: 0.0171
  train/loss_box_reg: 0.0442
  train/loss_objectness: 0.0005
  train/loss_rpn_box_reg: 0.0052
Best checkpoint: /home/florian/src/python/set-master/runs/card_detector/version_1/checkpoints/epoch-14.ckpt
Last checkpoint: /home/florian/src/python/set-master/runs/card_detector/version_1/checkpoints/last.ckpt


## Predictions

Ground truth is green; predictions are red.


In [ ]:
score_threshold = 0.75
if checkpoint.best_model_path:
    prediction_model = load_detector(Path(checkpoint.best_model_path), repo_path)
else:
    prediction_model = model.eval()

image, target = data.val_dataset[0]
with torch.inference_mode():
    prediction = prediction_model([image.to(prediction_model.device)])[0]
display_image(prediction_overlay(image, target, prediction, score_threshold))
